<a href="https://colab.research.google.com/github/Aksinhaa/ColabFold/blob/main/TGW_damageprofiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This workshop is a joint collaboration between NCBS - Bangalore, India `(Uma Ramakrishnan, Laura Bertola, Devkant Singha, Vinti Nanda)`; University of Edinburgh UK `(Katerina Guschanski, German Hernandez Alonso)`; IISc - Bangalore, India `(Anubhab Khan, Akancha Sinha )`; IISER Mohali, India `(Kritika M. Garg)`; and Birbal Sahni Institute of Palaeosciences (BSIP) - Lucknow, India `(Niraj Rai)`.
Scripts written by: `German Hernandez Alonso`
Notebook compiled by: `Akancha Sinha`





Ancient DNA (aDNA) analysis presents unique challenges compared to modern genomic data due to degradation processes that occur over time.

One of the most characteristic features of ancient DNA is the presence of **post-mortem damage patterns**, primarily caused by cytosine deamination. This biochemical process leads to an increased frequency of cytosine (C) to thymine (T) substitutions at the 5′ ends of DNA fragments and complementary guanine (G) to adenine (A) substitutions at the 3′ ends. These misincorporation patterns serve as molecular signatures that distinguish authentic ancient DNA from modern contamination.

In addition to nucleotide misincorporations, ancient DNA is typically characterized by **short fragment lengths**, reflecting extensive degradation over time. Together, these features—fragmentation and damage patterns—form the basis for computational authentication of ancient DNA datasets.

---



DamageProfiler is a specialized bioinformatics tool designed to quantify and visualize damage patterns in sequencing data. It analyzes aligned sequencing reads (BAM files) against a reference genome and computes:

* Misincorporation frequencies (C→T and G→A substitutions)
* DNA fragment length distributions
* Position-specific damage patterns along reads

The output includes both graphical plots and tabular summaries, enabling researchers to assess the authenticity and quality of ancient DNA samples.



In this notebook, we use DamageProfiler to:

* Evaluate nucleotide misincorporation patterns in aligned sequencing data
* Assess characteristic ancient DNA damage signatures
* Examine fragment length distributions
* Validate the authenticity of the dataset before downstream analysis




In [ ]:
%%bash
# Install Miniconda
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda

# Initialize conda
source /usr/local/miniconda/etc/profile.d/conda.sh

# Accept Terms of Service
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Add channels
conda config --add channels defaults
conda config --add channels bioconda
conda config --add channels conda-forge

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda create -y -n ngs_env damageprofiler

For this step of the analysis, we are working with a different chromosome (chromosome 26) and a corresponding BAM file instead of the dataset used in previous steps. The required BAM file, its index, and the reference genome files are not preloaded, so we use `wget` to download them directly from online repositories (Zenodo) into the Colab environment. This approach ensures that all necessary input files are available within the working directory and keeps the workflow fully reproducible within the notebook environment.


In [ ]:
%%bash

source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

mkdir -p /content/workshop_data/damage_profiler_chr26
cd  /content/workshop_data/damage_profiler_chr26

wget https://zenodo.org/records/19914378/files/APOR-002_chr26.bam
wget https://zenodo.org/records/19914378/files/APOR-002_chr26.bam.bai
wget https://zenodo.org/records/19947652/files/chr26_ovis.fasta
wget https://zenodo.org/records/20024985/files/chr26.fasta.amb
wget https://zenodo.org/records/20024985/files/chr26.fasta.ann
wget https://zenodo.org/records/20024985/files/chr26.fasta.bwt
wget https://zenodo.org/records/20024985/files/chr26.fasta.fai
wget https://zenodo.org/records/20024985/files/chr26.fasta.pac
wget https://zenodo.org/records/20024985/files/chr26.fasta.sa


This step analyzes post-mortem DNA damage patterns using DamageProfiler for each sample. The script loops through all samples and runs the tool on the filtered, deduplicated BAM files, using the reference genome for comparison.

The `-i` parameter specifies the input BAM file, while `-r` provides the reference genome needed to assess mismatches. The `-t 2` option allows multi-threading for faster processing, and `-o` defines the output directory where damage reports (HTML and plots) are generated. The `-yaxis_dp_max 0.30` sets the maximum y-axis limit for damage plots, making patterns easier to visualize.

This step is critical in ancient DNA analysis because it identifies characteristic damage signatures such as C→T substitutions at the 5′ ends and G→A at the 3′ ends of reads. These patterns help confirm the authenticity of aDNA and distinguish it from modern contamination. Overall, DamageProfiler provides both quantitative and visual evidence of DNA degradation, which is essential for validating downstream analyses.


In [ ]:
%%bash

source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/damage_profiler_chr26

echo "Fixing reference naming consistency..."

# Make DamageProfiler see the expected index filename
ln -s chr26.fasta.fai chr26_ovis.fasta.fai

echo "Running DamageProfiler..."

mkdir -p output_APOR_002

conda run -n ngs_env damageprofiler \
  -i APOR-002_chr26.bam \
  -r chr26_ovis.fasta \
  -t 2 \
  -o output_APOR_002 \
  -yaxis_dp_max 0.30

In [ ]:
!apt-get install -y poppler-utils

!pdftoppm /content/workshop_data/damage_profiler_chr26/output_APOR_002/DamagePlot.pdf damage_plot -png
!pdftoppm /content/workshop_data/damage_profiler_chr26/output_APOR_002/Length_plot.pdf length_plot -png

from IPython.display import Image, display

# Display both images
display(Image("damage_plot-1.png"))
display(Image("length_plot-1.png"))